# Age Comparison — Pairwise Learning con UTKFace

Questo notebook unifica **tutte le fasi del progetto** in un unico documento eseguibile:

1. Setup & imports
2. Dataset (`datasets.py`)
3. Modelli (`model.py`, `age_regression_model.py`, `pair_mlp_model.py`)
4. Utilità (`utils.py`)
5. Training — Fase 1: Age Regressor
6. Training — Fase 2: Pairwise MLP
7. Training — Early Fusion (ResNet 6 canali)
8. Inference

> **Dataset atteso:** cartella `data/UTKFace/` con immagini nominatE come `age_gender_race_timestamp.jpg`.

## 0. Setup & librerie

Installiamo le dipendenze mancanti e verifichiamo le versioni.

In [19]:
# Installa dipendenze se necessario
try:
    import torchvision
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "torch", "torchvision", "tensorboard", "Pillow"])

import torch
import torchvision

print(f"PyTorch     : {torch.__version__}")
print(f"torchvision : {torchvision.__version__}")
print(f"GPU disponibile: {torch.cuda.is_available()}")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device selezionato: {DEVICE}")

PyTorch     : 2.5.1+cu121
torchvision : 0.20.1+cu121
GPU disponibile: True
Device selezionato: cuda


## 1. Dataset — `datasets.py`

### 1.1 Parsing e raccolta dei file UTKFace

Il nome di ogni immagine UTKFace segue il pattern `age_gender_race_timestamp.jpg`.
`parse_age_from_filename` estrae l'età; `collect_utkface` scansiona la cartella.

In [20]:
import os
import random
from typing import List, Tuple, Optional

from PIL import Image
from torch.utils.data import Dataset
import torchvision.transforms as T


def parse_age_from_filename(filename: str) -> Optional[int]:
    """
    Estrae l'età da un nome file UTKFace del tipo:
    age_gender_race_time.jpg  (es. 25_0_0_20170116174525125.jpg).
    Ritorna None se il formato non è valido.
    """
    base = os.path.basename(filename)
    name, _ = os.path.splitext(base)
    parts = name.split("_")
    if len(parts) < 4:
        return None
    try:
        return int(parts[0])
    except ValueError:
        return None


def collect_utkface(root_dir: str) -> List[Tuple[str, int]]:
    """
    Scansiona la cartella UTKFace e ritorna una lista di tuple
    (percorso_assoluto_immagine, eta).
    """
    imgs: List[Tuple[str, int]] = []
    for fname in os.listdir(root_dir):
        if not fname.lower().endswith((".jpg", ".jpeg", ".png")):
            continue
        age = parse_age_from_filename(fname)
        if age is None:
            continue
        imgs.append((os.path.join(root_dir, fname), age))

    if not os.path.isdir(UTK_ROOT):
        print(f"⚠️  Cartella non trovata: {UTK_ROOT} — skippo la verifica.")

    if len(imgs) < 2:
        raise RuntimeError("Troppo poche immagini valide trovate in UTKFace.")

    return imgs

### 1.2 `SingleAgeDataset` — un'immagine per campione (age regression)

In [21]:
class SingleAgeDataset(Dataset):
    """
    Dataset UTKFace per stima dell'età (age regression).
    Ogni elemento è (img_tensor, age_float).
    """

    def __init__(self, samples: List[Tuple[str, int]], transform=None):
        self.samples = samples
        self.transform = transform or T.Compose([
            T.Resize((224, 224)),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406],
                        std =[0.229, 0.224, 0.225]),
        ])

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, index: int):
        path, age = self.samples[index]
        img = Image.open(path).convert("RGB")
        img = self.transform(img)
        return img, float(age)

### 1.3 `PairwiseAgeDataset` — coppie di immagini per il confronto di età

In [22]:
class PairwiseAgeDataset(Dataset):
    """
    Dataset che genera coppie di facce per il confronto di età.

    Ogni elemento è (img1, img2, label) dove:
      - label = 0  se img1 è più giovane di img2  (age1 < age2)
      - label = 1  altrimenti

    min_age_gap garantisce che le coppie siano 'facili' da distinguere.
    """

    def __init__(
        self,
        samples: List[Tuple[str, int]],
        transform=None,
        min_age_gap: int = 10,
        seed: Optional[int] = None,
    ):
        self.samples      = samples
        self.min_age_gap  = int(min_age_gap)
        self.transform    = transform or T.Compose([
            T.Resize((224, 224)),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406],
                        std =[0.229, 0.224, 0.225]),
        ])
        self.rng = random.Random(seed)

        if len(self.samples) < 2:
            raise ValueError("Servono almeno 2 immagini per creare coppie.")

    def __len__(self) -> int:
        return len(self.samples) * 2          # numero virtuale di coppie

    def _sample_two_indices(self) -> Tuple[int, int]:
        n, max_tries = len(self.samples), 1000
        for _ in range(max_tries):
            i1 = self.rng.randrange(n)
            i2 = self.rng.randrange(n)
            if i2 == i1:
                continue
            age1, age2 = self.samples[i1][1], self.samples[i2][1]
            if abs(age1 - age2) >= self.min_age_gap:
                return i1, i2
        raise RuntimeError(
            f"Impossibile campionare una coppia con differenza >= {self.min_age_gap}. "
            "Prova a ridurre min_age_gap."
        )

    def __getitem__(self, index: int):
        i1, i2             = self._sample_two_indices()
        path1, age1        = self.samples[i1]
        path2, age2        = self.samples[i2]
        img1               = self.transform(Image.open(path1).convert("RGB"))
        img2               = self.transform(Image.open(path2).convert("RGB"))
        label              = 0 if age1 < age2 else 1
        return img1, img2, label


# Alias semantico
AgeComparisonDataset = PairwiseAgeDataset

### 1.4 Verifica rapida del dataset

In [23]:
UTK_ROOT = os.path.join("data", "UTKFace")   # ← adatta se necessario

samples = collect_utkface(UTK_ROOT)
ages    = [age for _, age in samples]

print(f"Immagini totali : {len(samples)}")
print(f"Età min/max     : {min(ages)} / {max(ages)}")
print(f"Età media       : {sum(ages)/len(ages):.1f}")
print(f"Esempio         : {samples[0]}")

Immagini totali : 24103
Età min/max     : 1 / 116
Età media       : 33.0
Esempio         : ('data/UTKFace/23_1_3_20170104231800546.jpg', 23)


## 2. Modelli

Definiamo i tre moduli usati nel progetto.

### 2.1 `ResNetPairwiseAge` — Early Fusion (6 canali)

Le due immagini vengono concatenate lungo i canali prima di entrare nella ResNet18.

In [24]:
from typing import cast
import torch
import torch.nn as nn
from torchvision.models import resnet18


class ResNetPairwiseAge(nn.Module):
    """
    ResNet18 con primo conv modificato da 3→6 canali.
    Riceve (img1, img2) e produce 2 logit:
      0 = img1 più giovane  |  1 = img1 più vecchia (o coetanea).
    """

    def __init__(self, pretrained: bool = True) -> None:
        super().__init__()
        weights = "IMAGENET1K_V1" if pretrained else None
        base    = resnet18(weights=weights)

        old_conv = base.conv1
        new_conv = nn.Conv2d(
            in_channels  = 6,
            out_channels = old_conv.out_channels,
            kernel_size  = cast(tuple[int, int], old_conv.kernel_size),
            stride       = cast(tuple[int, int], old_conv.stride),
            padding      = cast(tuple[int, int] | str, old_conv.padding),
            bias         = old_conv.bias is not None,
        )
        with torch.no_grad():
            new_conv.weight[:, :3, :, :] = old_conv.weight
            new_conv.weight[:, 3:, :, :] = old_conv.weight
        base.conv1 = new_conv

        in_features = base.fc.in_features
        base.fc     = nn.Linear(in_features, 2)
        self.backbone = base

    def forward(self, img1: torch.Tensor, img2: torch.Tensor) -> torch.Tensor:
        x = torch.cat([img1, img2], dim=1)
        return self.backbone(x)

### 2.2 `AgeResNet18` — Regressore età (scalare)

In [25]:
import copy
class AgeResNet18(nn.Module):
    """
    ResNet18 con testa di regressione scalare per predire l'età.
    """

    def __init__(self, pretrained: bool = True) -> None:
        super().__init__()
        weights       = "IMAGENET1K_V1" if pretrained else None
        self.backbone = resnet18(weights=weights)
        in_features   = self.backbone.fc.in_features
        self.backbone.fc = nn.Linear(in_features, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.backbone(x).squeeze(1)   # shape: [B]


class AgeFeatureExtractor(nn.Module):
    """
    Wrapper che rimuove la testa di regressione da AgeResNet18
    e restituisce il vettore di embedding [B, out_dim].
    """

    def __init__(self, pretrained_backbone: AgeResNet18) -> None:
        super().__init__()
        backbone         = copy.deepcopy(pretrained_backbone.backbone)
        in_features      = backbone.fc.in_features
        backbone.fc      = nn.Identity()
        self.backbone    = backbone
        self.out_dim     = in_features

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.backbone(x)

### 2.3 `PairwiseAgeMLP` — Testa MLP per confronto pairwise

In [26]:
class PairwiseAgeMLP(nn.Module):
    """
    MLP che riceve concat(f1, f2, |f1−f2|) e predice chi è più giovane.
    Input  : 3 × feature_dim
    Output : 2 logit (classe 0/1)
    """

    def __init__(self, feature_dim: int, hidden_dim: int = 256) -> None:
        super().__init__()
        in_dim   = feature_dim * 3
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),
            nn.Linear(hidden_dim, 2),
        )

    def forward(self, f1: torch.Tensor, f2: torch.Tensor) -> torch.Tensor:
        diff = torch.abs(f1 - f2)
        x    = torch.cat([f1, f2, diff], dim=1)
        return self.net(x)

## 3. Utilità — `utils.py`

Funzioni di logging (TensorBoard) e gestione dei checkpoint.

In [27]:
import datetime
from torch.utils.tensorboard import SummaryWriter


def create_writer(experiment_name: str, config_name: str) -> SummaryWriter:
    """
    Crea un SummaryWriter con timestamp.
    Percorso esempio: runs/pairwise_resnet/lr1e-4_bs64_20240101_120000/
    """
    timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    log_dir   = os.path.join("runs", experiment_name, f"{config_name}_{timestamp}")
    writer    = SummaryWriter(log_dir=log_dir)
    print(f"[INFO] TensorBoard log dir: {log_dir}")
    return writer


def save_checkpoint(
    state: dict,
    is_best: bool,
    save_dir: str,
    filename: str      = "last_checkpoint.pth",
    best_filename: str = "best_model.pth",
) -> None:
    """Salva sempre l'ultimo checkpoint; se is_best, salva anche la copia migliore."""
    os.makedirs(save_dir, exist_ok=True)
    last_path = os.path.join(save_dir, filename)
    torch.save(state, last_path)
    if is_best:
        best_path = os.path.join(save_dir, best_filename)
        torch.save(state, best_path)
        print(f"  ✅ New best → {best_path}")


def load_checkpoint(
    resume_path: str,
    model: nn.Module,
    optimizer: torch.optim.Optimizer = None,
    device: str = "cpu",
) -> Tuple[int, float]:
    """Carica pesi, ottimizzatore ed epoca da un file .pth."""
    if not os.path.isfile(resume_path):
        print(f"=> Nessun checkpoint in '{resume_path}'. Partenza da zero.")
        return 1, float('inf')

    print(f"=> Caricamento checkpoint da '{resume_path}'")
    checkpoint = torch.load(resume_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])

    start_epoch    = checkpoint.get('epoch', 0) + 1
    best_val_loss  = checkpoint.get('best_val_loss', float('inf'))

    if optimizer and 'optimizer_state_dict' in checkpoint:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

    print(f"=> Ripresa dall'epoca {start_epoch}")
    return start_epoch, best_val_loss

## 4. Training — Fase 1: Age Regressor

`AgeResNet18` viene addestrata su UTKFace per predire l'età come valore continuo (regressione MSE).
I pesi risultanti costituiscono il feature extractor per la Fase 2.

### 4.1 DataLoader Fase 1

In [28]:
from torch.utils.data import DataLoader, random_split
import torch.optim as optim


def make_dataloaders_regression(
    utk_root: str,
    batch_size: int  = 64,
    val_split: float = 0.1,
) -> Tuple[DataLoader, DataLoader]:

    samples = collect_utkface(utk_root)
    n_val   = max(1, int(len(samples) * val_split))
    n_train = len(samples) - n_val
    train_s, val_s = random_split(samples, [n_train, n_val])

    train_transform = T.Compose([
        T.Resize((224, 224)),
        T.RandomHorizontalFlip(p=0.5),
        T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    val_transform = T.Compose([
        T.Resize((224, 224)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    train_ds = SingleAgeDataset(list(train_s), transform=train_transform)
    val_ds   = SingleAgeDataset(list(val_s),   transform=val_transform)

    return (
        DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=4),
        DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=4),
    )

### 4.2 Loop di addestramento Fase 1

In [29]:
def train_age_regressor(
    utk_root:    str,
    epochs:      int   = 10,
    lr:          float = 1e-4,
    batch_size:  int   = 64,
    device:      str   = DEVICE,
    save_dir:    str   = "models",
    resume_path: str   = None,
) -> AgeResNet18:

    train_loader, val_loader = make_dataloaders_regression(utk_root, batch_size)

    model     = AgeResNet18(pretrained=True).to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    start_epoch, best_val_loss = 1, float('inf')
    if resume_path:
        start_epoch, best_val_loss = load_checkpoint(resume_path, model, optimizer, device)

    writer = create_writer("age_regressor", f"lr{lr}_bs{batch_size}")

    for epoch in range(start_epoch, epochs + 1):
        # ── TRAIN ──────────────────────────────────────────────────────────
        model.train()
        train_loss = train_mae = n_train = 0

        for imgs, ages in train_loader:
            imgs, ages = imgs.to(device), ages.to(device).float()
            optimizer.zero_grad()
            preds = model(imgs)
            loss  = criterion(preds, ages)
            loss.backward()
            optimizer.step()

            bs           = ages.size(0)
            train_loss  += loss.item() * bs
            train_mae   += torch.abs(preds - ages).sum().item()
            n_train     += bs

        train_loss /= n_train
        train_mae  /= n_train

        # ── VALIDATION ─────────────────────────────────────────────────────
        model.eval()
        val_loss = val_mae = n_val = 0
        with torch.no_grad():
            for imgs, ages in val_loader:
                imgs, ages = imgs.to(device), ages.to(device).float()
                preds      = model(imgs)
                loss       = criterion(preds, ages)
                bs         = ages.size(0)
                val_loss  += loss.item() * bs
                val_mae   += torch.abs(preds - ages).sum().item()
                n_val     += bs

        val_loss /= n_val
        val_mae  /= n_val

        print(
            f"[AgeReg] Epoch {epoch:>3}/{epochs} "
            f"| train MSE={train_loss:.4f}  MAE={train_mae:.2f}y "
            f"| val MSE={val_loss:.4f}  MAE={val_mae:.2f}y"
        )

        writer.add_scalar('Loss/Train_MSE',        train_loss, epoch)
        writer.add_scalar('Loss/Validation_MSE',   val_loss,   epoch)
        writer.add_scalar('Error/Train_MAE_Years',  train_mae,  epoch)
        writer.add_scalar('Error/Val_MAE_Years',    val_mae,    epoch)

        is_best = val_loss < best_val_loss
        if is_best:
            best_val_loss = val_loss

        save_checkpoint(
            {
                'epoch': epoch, 'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': val_loss, 'best_val_loss': best_val_loss,
            },
            is_best, save_dir,
            filename="last_age_reg.pth", best_filename="best_age_reg.pth",
        )

    writer.close()
    print("\n✅ Fase 1 completata!")
    return model

### 4.3 Avvio training Fase 1

Modifica i parametri sotto prima di eseguire.

In [30]:
# ── Iperparametri Fase 1 ─────────────────────────────────────────────────
AGE_REG_EPOCHS     = 10
AGE_REG_LR         = 1e-4
AGE_REG_BATCH_SIZE = 64
AGE_REG_SAVE_DIR   = "models"
AGE_REG_RESUME     = None   # es. "models/last_age_reg.pth"

age_reg_model = train_age_regressor(
    utk_root    = UTK_ROOT,
    epochs      = AGE_REG_EPOCHS,
    lr          = AGE_REG_LR,
    batch_size  = AGE_REG_BATCH_SIZE,
    device      = DEVICE,
    save_dir    = AGE_REG_SAVE_DIR,
    resume_path = AGE_REG_RESUME,
)

[INFO] TensorBoard log dir: runs/age_regressor/lr0.0001_bs64_20260830_204211


KeyboardInterrupt: 

## 5. Training — Fase 2: Pairwise MLP

Il CNN backbone (fase 1) viene **congelato**; si addestra solo la testa MLP
che riceve `concat(f1, f2, |f1−f2|)`.

### 5.1 DataLoader Fase 2

In [ ]:
def make_dataloaders_mlp(
    utk_root:  str,
    batch_size: int  = 64,
    val_split:  float = 0.1,
) -> Tuple[DataLoader, DataLoader]:

    samples = collect_utkface(utk_root)
    n_val   = max(1, int(len(samples) * val_split))
    n_train = len(samples) - n_val
    train_s, val_s = random_split(samples, [n_train, n_val])

    train_ds = PairwiseAgeDataset(list(train_s), min_age_gap=10, seed=123)
    val_ds   = PairwiseAgeDataset(list(val_s),   min_age_gap=10, seed=124)

    return (
        DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=4),
        DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=4),
    )

### 5.2 Loop di addestramento Fase 2

In [ ]:
def train_pairwise_mlp(
    utk_root:          str,
    age_regressor_ckpt: str,
    epochs:            int   = 5,
    lr:                float = 1e-5,
    batch_size:        int   = 64,
    device:            str   = DEVICE,
    save_dir:          str   = "models",
    resume_path:       str   = None,
    age_reg_model:     AgeResNet18 = None,   # opzionale: modello già in memoria
) -> PairwiseAgeMLP:

    # ── Feature extractor (congelato) ────────────────────────────────────
    if age_reg_model is None:
        age_reg_model = AgeResNet18(pretrained=False)
        ckpt = torch.load(age_regressor_ckpt, map_location="cpu")
        state = ckpt.get('model_state_dict', ckpt)
        age_reg_model.load_state_dict(state)

    feature_extractor = AgeFeatureExtractor(age_reg_model).to(device)
    feature_extractor.eval()
    for p in feature_extractor.parameters():
        p.requires_grad = False

    # ── MLP ──────────────────────────────────────────────────────────────
    mlp       = PairwiseAgeMLP(feature_dim=feature_extractor.out_dim).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(mlp.parameters(), lr=lr, weight_decay=1e-4)

    start_epoch, best_val_loss = 1, float('inf')
    if resume_path:
        start_epoch, best_val_loss = load_checkpoint(resume_path, mlp, optimizer, device)

    writer = create_writer("pairwise_mlp", f"lr{lr}_bs{batch_size}")
    train_loader, val_loader = make_dataloaders_mlp(utk_root, batch_size)

    for epoch in range(start_epoch, epochs + 1):
        # ── TRAIN ──────────────────────────────────────────────────────────
        mlp.train()
        train_loss = correct = total = 0

        for img1, img2, labels in train_loader:
            img1, img2, labels = img1.to(device), img2.to(device), labels.to(device)
            with torch.no_grad():
                f1, f2 = feature_extractor(img1), feature_extractor(img2)

            optimizer.zero_grad()
            logits = mlp(f1, f2)
            loss   = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            bs          = labels.size(0)
            train_loss += loss.item() * bs
            correct    += (logits.argmax(1) == labels).sum().item()
            total      += bs

        train_loss /= total
        train_acc   = correct / total

        # ── VALIDATION ─────────────────────────────────────────────────────
        mlp.eval()
        val_loss = val_correct = val_total = 0
        with torch.no_grad():
            for img1, img2, labels in val_loader:
                img1, img2, labels = img1.to(device), img2.to(device), labels.to(device)
                f1, f2  = feature_extractor(img1), feature_extractor(img2)
                logits  = mlp(f1, f2)
                loss    = criterion(logits, labels)
                bs      = labels.size(0)
                val_loss    += loss.item() * bs
                val_correct += (logits.argmax(1) == labels).sum().item()
                val_total   += bs

        val_loss /= val_total
        val_acc   = val_correct / val_total

        print(
            f"[MLP] Epoch {epoch:>3}/{epochs} "
            f"| train loss={train_loss:.4f}  acc={train_acc:.4f} "
            f"| val loss={val_loss:.4f}  acc={val_acc:.4f}"
        )

        writer.add_scalar('Loss/Train',         train_loss, epoch)
        writer.add_scalar('Loss/Validation',    val_loss,   epoch)
        writer.add_scalar('Accuracy/Train',     train_acc,  epoch)
        writer.add_scalar('Accuracy/Validation',val_acc,    epoch)

        is_best = val_loss < best_val_loss
        if is_best:
            best_val_loss = val_loss

        save_checkpoint(
            {
                'epoch': epoch, 'model_state_dict': mlp.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': val_loss, 'best_val_loss': best_val_loss,
            },
            is_best, save_dir,
            filename="last_mlp.pth", best_filename="best_mlp.pth",
        )

    writer.close()
    print("\n✅ Fase 2 (MLP) completata!")
    return mlp

### 5.3 Avvio training Fase 2

Assicurati che `best_age_reg.pth` esista (output Fase 1).

In [ ]:
# ── Iperparametri Fase 2 ─────────────────────────────────────────────────
MLP_EPOCHS     = 5
MLP_LR         = 1e-5
MLP_BATCH_SIZE = 64
MLP_SAVE_DIR   = "models"
MLP_RESUME     = None   # es. "models/last_mlp.pth"
AGE_REG_CKPT   = os.path.join("models", "best_age_reg.pth")

mlp_model = train_pairwise_mlp(
    utk_root           = UTK_ROOT,
    age_regressor_ckpt = AGE_REG_CKPT,
    epochs             = MLP_EPOCHS,
    lr                 = MLP_LR,
    batch_size         = MLP_BATCH_SIZE,
    device             = DEVICE,
    save_dir           = MLP_SAVE_DIR,
    resume_path        = MLP_RESUME,
    age_reg_model      = age_reg_model,   # riusa il modello già in memoria
)

[INFO] TensorBoard log dir: runs/pairwise_mlp/lr1e-05_bs64_20260830_201132
[MLP] Epoch   1/5 | train loss=0.1411  acc=0.9768 | val loss=0.0407  acc=0.9902
  ✅ New best → models/best_mlp.pth
[MLP] Epoch   2/5 | train loss=0.0380  acc=0.9900 | val loss=0.0321  acc=0.9911
  ✅ New best → models/best_mlp.pth
[MLP] Epoch   3/5 | train loss=0.0330  acc=0.9905 | val loss=0.0291  acc=0.9902
  ✅ New best → models/best_mlp.pth
[MLP] Epoch   4/5 | train loss=0.0317  acc=0.9908 | val loss=0.0282  acc=0.9902
  ✅ New best → models/best_mlp.pth
[MLP] Epoch   5/5 | train loss=0.0309  acc=0.9910 | val loss=0.0272  acc=0.9934
  ✅ New best → models/best_mlp.pth

✅ Fase 2 (MLP) completata!


## 6. Training — Early Fusion (ResNet 6 canali)

`ResNetPairwiseAge` riceve le due immagini concatenate lungo i canali
(approccio **end-to-end**, senza CNN congelata).

### 6.1 DataLoader Early Fusion

In [ ]:
def make_dataloaders_early_fusion(
    utk_root:  str,
    batch_size: int  = 64,
    val_split:  float = 0.1,
) -> Tuple[DataLoader, DataLoader]:

    samples = collect_utkface(utk_root)
    n_val   = max(1, int(len(samples) * val_split))
    n_train = len(samples) - n_val
    train_s, val_s = random_split(samples, [n_train, n_val])

    train_transform = T.Compose([
        T.Resize((224, 224)),
        T.RandomHorizontalFlip(p=0.5),
        T.RandomRotation(degrees=10),
        T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    eval_transform = T.Compose([
        T.Resize((224, 224)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    train_ds = PairwiseAgeDataset(list(train_s), transform=train_transform, min_age_gap=10, seed=42)
    val_ds   = PairwiseAgeDataset(list(val_s),   transform=eval_transform,  min_age_gap=10, seed=43)

    return (
        DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=4),
        DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=4),
    )

### 6.2 Loop di addestramento Early Fusion

In [ ]:
def train_early_fusion(
    utk_root:    str,
    epochs:      int   = 30,
    lr:          float = 1e-4,
    batch_size:  int   = 64,
    device:      str   = DEVICE,
    save_dir:    str   = "models",
    resume_path: str   = None,
) -> ResNetPairwiseAge:

    train_loader, val_loader = make_dataloaders_early_fusion(utk_root, batch_size)

    model     = ResNetPairwiseAge(pretrained=True).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)

    start_epoch, best_val_loss = 1, float('inf')
    if resume_path:
        start_epoch, best_val_loss = load_checkpoint(resume_path, model, optimizer, device)

    writer = create_writer("pairwise_resnet", f"lr{lr}_bs{batch_size}")

    for epoch in range(start_epoch, epochs + 1):
        # ── TRAIN ──────────────────────────────────────────────────────────
        model.train()
        running_loss = correct = total = 0

        for img1, img2, labels in train_loader:
            img1, img2, labels = img1.to(device), img2.to(device), labels.to(device)
            optimizer.zero_grad()
            logits = model(img1, img2)
            loss   = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            bs            = labels.size(0)
            running_loss += loss.item() * bs
            correct      += (logits.argmax(1) == labels).sum().item()
            total        += bs

        train_loss = running_loss / total
        train_acc  = correct / total

        # ── VALIDATION ─────────────────────────────────────────────────────
        model.eval()
        val_loss = val_correct = val_total = 0
        with torch.no_grad():
            for img1, img2, labels in val_loader:
                img1, img2, labels = img1.to(device), img2.to(device), labels.to(device)
                logits  = model(img1, img2)
                loss    = criterion(logits, labels)
                bs      = labels.size(0)
                val_loss    += loss.item() * bs
                val_correct += (logits.argmax(1) == labels).sum().item()
                val_total   += bs

        val_loss /= val_total
        val_acc   = val_correct / val_total

        print(
            f"[EarlyFusion] Epoch {epoch:>3}/{epochs} "
            f"| train loss={train_loss:.4f}  acc={train_acc:.4f} "
            f"| val loss={val_loss:.4f}  acc={val_acc:.4f}"
        )

        writer.add_scalar('Loss/Train',          train_loss, epoch)
        writer.add_scalar('Loss/Validation',     val_loss,   epoch)
        writer.add_scalar('Accuracy/Train',      train_acc,  epoch)
        writer.add_scalar('Accuracy/Validation', val_acc,    epoch)

        is_best = val_loss < best_val_loss
        if is_best:
            best_val_loss = val_loss

        save_checkpoint(
            {
                'epoch': epoch, 'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': val_loss, 'best_val_loss': best_val_loss,
            },
            is_best, save_dir,
            filename="last_pairwise.pth", best_filename="best_pairwise.pth",
        )

    writer.close()
    print("\n✅ Early Fusion completato!")
    return model

### 6.3 Avvio training Early Fusion

In [31]:
# ── Iperparametri Early Fusion ────────────────────────────────────────────
EF_EPOCHS     = 30
EF_LR         = 1e-4
EF_BATCH_SIZE = 64
EF_SAVE_DIR   = "models"
EF_RESUME     = None   # es. "models/last_pairwise.pth"

ef_model = train_early_fusion(
    utk_root    = UTK_ROOT,
    epochs      = EF_EPOCHS,
    lr          = EF_LR,
    batch_size  = EF_BATCH_SIZE,
    device      = DEVICE,
    save_dir    = EF_SAVE_DIR,
    resume_path = EF_RESUME,
)

[INFO] TensorBoard log dir: runs/pairwise_resnet/lr0.0001_bs64_20260830_204227
[EarlyFusion] Epoch   1/30 | train loss=0.5987  acc=0.6731 | val loss=0.5476  acc=0.7112
  ✅ New best → models/best_pairwise.pth
[EarlyFusion] Epoch   2/30 | train loss=0.4617  acc=0.7746 | val loss=0.4841  acc=0.7672
  ✅ New best → models/best_pairwise.pth
[EarlyFusion] Epoch   3/30 | train loss=0.3581  acc=0.8369 | val loss=0.5686  acc=0.7568
[EarlyFusion] Epoch   4/30 | train loss=0.2752  acc=0.8831 | val loss=0.6750  acc=0.7494
[EarlyFusion] Epoch   5/30 | train loss=0.1972  acc=0.9192 | val loss=0.7844  acc=0.7622
[EarlyFusion] Epoch   6/30 | train loss=0.1467  acc=0.9424 | val loss=0.8578  acc=0.7680
[EarlyFusion] Epoch   7/30 | train loss=0.1059  acc=0.9593 | val loss=0.8293  acc=0.7479
[EarlyFusion] Epoch   8/30 | train loss=0.0862  acc=0.9673 | val loss=0.7703  acc=0.7944
[EarlyFusion] Epoch   9/30 | train loss=0.0678  acc=0.9748 | val loss=0.7913  acc=0.7898
[EarlyFusion] Epoch  10/30 | train loss=

## 7. Inference

Confronta due foto e determina quale persona è più giovane.
Supporta entrambi i modelli (Siamese MLP e Early Fusion).

### 7.1 Funzioni di caricamento e predizione

In [32]:
def get_transform():
    return T.Compose([
        T.Resize((224, 224)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])


def load_model_from_ckpt(ckpt_path: str, device: str):
    """
    Carica il modello corretto in base al nome del checkpoint.
    Ritorna (model, mode, feature_extractor).
    """
    def _safe_load(model, path):
        ckpt  = torch.load(path, map_location=device)
        state = ckpt.get('model_state_dict', ckpt)
        model.load_state_dict(state)
        return model

    if "mlp" in ckpt_path:
        reg_path          = os.path.join("models", "best_age_reg.pth")
        regressor         = _safe_load(AgeResNet18(pretrained=False), reg_path)
        feature_extractor = AgeFeatureExtractor(regressor).to(device)
        mlp               = _safe_load(
            PairwiseAgeMLP(feature_dim=feature_extractor.out_dim).to(device),
            ckpt_path,
        )
        return mlp, "siamese", feature_extractor
    else:
        model = _safe_load(ResNetPairwiseAge(pretrained=False).to(device), ckpt_path)
        return model, "early_fusion", None


def infer(img_path1: str, img_path2: str, ckpt_path: str, device: str = DEVICE) -> str:
    """
    Confronta img_path1 e img_path2.
    Ritorna 'PIÙ GIOVANE' se la prima foto è più giovane, 'PIÙ VECCHIA' altrimenti.
    """
    model, mode, feature_extractor = load_model_from_ckpt(ckpt_path, device)
    model.eval()
    transform = get_transform()

    img1 = transform(Image.open(img_path1).convert("RGB")).unsqueeze(0).to(device)
    img2 = transform(Image.open(img_path2).convert("RGB")).unsqueeze(0).to(device)

    with torch.no_grad():
        if mode == "siamese":
            f1, f2  = feature_extractor(img1), feature_extractor(img2)
            logits  = model(f1, f2)
        else:
            logits  = model(img1, img2)

    pred   = logits.argmax(dim=1).item()
    result = "PIÙ GIOVANE" if pred == 0 else "PIÙ VECCHIA"
    print(f"Modello: {mode} | La prima foto è {result} della seconda")
    return result

### 7.2 Esempio di inference

Modifica i percorsi delle immagini e del checkpoint.

In [33]:
# ── Parametri Inference ──────────────────────────────────────────────────
IMG1      = "data/UTKFace/foto_flavio.jpg"   # ← sostituisci
IMG2      = "data/UTKFace/58_0_0_20170117191847355.jpg"   # ← sostituisci
CKPT_PATH = "models/best_pairwise.pth"    # oppure "models/best_mlp.pth"

# Esegui solo se i file esistono
if os.path.exists(IMG1) and os.path.exists(IMG2) and os.path.exists(CKPT_PATH):
    result = infer(IMG1, IMG2, CKPT_PATH)
else:
    print("⚠️  Imposta percorsi validi per IMG1, IMG2 e CKPT_PATH prima di eseguire.")

/tmp/ipykernel_1426/949962212.py:15: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt  = torch.load(path, map_location=device)


Modello: early_fusion | La prima foto è PIÙ GIOVANE della seconda


## 8. Visualizzazione TensorBoard

Esegui la cella seguente per avviare TensorBoard inline (funziona su Google Colab e Jupyter Lab).

In [34]:
%load_ext tensorboard
%tensorboard --logdir runs

Reusing TensorBoard on port 6006 (pid 16067), started 1:59:40 ago. (Use '!kill 16067' to kill it.)